# 項目1: テーマ・エクスポージャの構築

この notebook は Phase B の項目1だけを検証するための切り出しである。目的は、point-in-time のテーマバスケット構成と日次Barraエクスポージャから、銘柄別テーマ構成行列 `A_t` とテーマ・エクスポージャ行列 `X_M,t` を作ることに限定する。

ここでは Barra 既知因子からの純化 `Z = X_{M\perp V}`、OLS係数、OLS残差、adjusted R²、テーマリターン推定、コヒーレンス判定、最適化は扱わない。それらは項目2以降で確認する。


## 必要データ形式

この notebook は手元の実データのみで実行する。合成データは含めない。

| 入力 | 形式 | 必須内容 |
|---|---|---|
| `stock_returns` | wide | index=営業日、columns=`security_id`、values=日次トータルリターン |
| `theme_basket_weights` | long | `effective_date`, `theme_id`, `security_id`, `weight`。`available_date` は任意 |
| `barra_estimation_universe` | CSV long | `date`, `security_id`, `in_universe` |
| `data/temp/X_YYYYMMDD.pkl` | pickle | `{"date": Timestamp, "X": DataFrame(index=TSCODE, columns=factors)}` |
| `data/mapping.pkl` | pickle wide | index=`date`, columns=4桁コード、values=`TSCODE` |

`security_id` は `1234.T` 形式へ正規化する。`mapping.pkl` の当日行を使って `TSCODE -> 4桁コード -> security_id` の逆引きを作る。mappingの直近日forward-fillは行わず、当日行がなければ停止する。

OLS前提なので、外部の `barra_regression_weights` は使わない。有効銘柄すべての回帰ウェイトを `1.0` として、`X_M,t` は `sqrt(sum_i x_i^2)=1` になるよう標準化する。

期待する主な出力は次の4つである。

| 出力 | 内容 |
|---|---|
| `A_t` | 実装可能なテーマバスケット構成行列。列合計は1 |
| `coverage` | Universe・returns・mapping・Barraエクスポージャで残った各テーマ構成ウェイトの比率 |
| `X_M_t` | `theme_exposure_mode` に従って作成し、OLSノルムで単位化したテーマ・エクスポージャ |
| `diagnostics` | 使用ファイル、mapping、銘柄フィルタ、テーマフィルタ、列合計、ノルムの確認表 |


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Literal, Mapping, Sequence, Tuple
import math

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 180)


@dataclass
class ThemeExposureConfig:
    asof_date: str = "YYYY-MM-DD"
    theme_exposure_mode: Literal["basket_weight", "membership", "rank_weight"] = "basket_weight"
    min_basket_coverage: float = 0.80
    x_dir: str = "data/temp"
    mapping_path: str = "data/mapping.pkl"

    def validate(self) -> None:
        if self.asof_date == "YYYY-MM-DD":
            raise ValueError("cfg.asof_date を実データで確認したい基準日に変更してください。")
        if self.theme_exposure_mode not in {"basket_weight", "membership", "rank_weight"}:
            raise ValueError("theme_exposure_mode must be basket_weight, membership, or rank_weight.")
        if not 0 <= self.min_basket_coverage <= 1:
            raise ValueError("min_basket_coverage must lie in [0, 1].")


cfg = ThemeExposureConfig(
    asof_date="YYYY-MM-DD",
    theme_exposure_mode="basket_weight",
    min_basket_coverage=0.80,
    x_dir="data/temp",
    mapping_path="data/mapping.pkl",
)

paths = {
    "stock_returns": "data/stock_returns.csv",
    "theme_basket_weights": "data/theme_basket_weights.csv",
    "barra_estimation_universe": "data/barra_estimation_universe.csv",
}


## データ読込・正規化

wide型CSVは、`date` 列があればそれをindexにする。`date` 列がない場合は先頭列を日付列として扱う。`stock_returns` とテーマ構成、Universeの `security_id` は `1234.T` 形式へ正規化する。


In [ ]:
def _read_table(path: str | Path, wide: bool = False) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)

    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt"}:
        df = pd.read_csv(path)
    elif suffix in {".pkl", ".pickle"}:
        df = pd.read_pickle(path)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    elif suffix == ".feather":
        df = pd.read_feather(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    if wide:
        date_col = "date" if "date" in df.columns else df.columns[0]
        df[date_col] = pd.to_datetime(df[date_col])
        df = df.set_index(date_col)
    return df


def _to_naive_timestamp(value: Any) -> pd.Timestamp:
    ts = pd.Timestamp(value)
    if ts.tzinfo is not None:
        ts = ts.tz_localize(None)
    return ts.normalize()


def _format_code4(value: Any) -> str:
    if pd.isna(value):
        raise ValueError("4桁コードにNaNは使えません。")
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    if text.endswith(".T"):
        text = text[:-2]
    return text.zfill(4)


def _normalise_security_id(value: Any) -> str:
    code = _format_code4(value)
    return f"{code}.T"


def _normalise_tscode(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip()


def _normalise_wide(df: pd.DataFrame, name: str) -> pd.DataFrame:
    out = df.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index)).tz_localize(None).normalize()
    out = out[~out.index.duplicated(keep="last")].sort_index()
    out.columns = [_normalise_security_id(c) for c in out.columns]
    if pd.Index(out.columns).duplicated().any():
        dup = pd.Index(out.columns)[pd.Index(out.columns).duplicated()].unique().tolist()
        raise ValueError(f"{name} columns are duplicated after security_id normalization: {dup[:10]}")
    out = out.apply(pd.to_numeric, errors="coerce")
    if out.empty:
        raise ValueError(f"{name} is empty.")
    return out.astype(float)


def _normalise_long_dates(df: pd.DataFrame, date_cols: Sequence[str]) -> pd.DataFrame:
    out = df.copy()
    for col in date_cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col]).dt.tz_localize(None).dt.normalize()
    return out


def _coerce_bool(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce").fillna(0.0) != 0.0
    return series.astype(str).str.strip().str.lower().isin({"true", "t", "1", "yes", "y"})


def load_mapping(path: str | Path) -> pd.DataFrame:
    mapping = pd.read_pickle(path)
    if not isinstance(mapping, pd.DataFrame):
        raise TypeError("data/mapping.pkl must be a pandas DataFrame.")
    out = mapping.copy()
    out.index = pd.DatetimeIndex(pd.to_datetime(out.index)).tz_localize(None).normalize()
    out = out.sort_index()
    out.columns = [_format_code4(c) for c in out.columns]
    if pd.Index(out.columns).duplicated().any():
        dup = pd.Index(out.columns)[pd.Index(out.columns).duplicated()].unique().tolist()
        raise ValueError(f"mapping.pkl columns are duplicated after 4-digit normalization: {dup[:10]}")
    return out


def mapping_pairs_for_date(mapping: pd.DataFrame, asof_date: pd.Timestamp) -> pd.DataFrame:
    asof = _to_naive_timestamp(asof_date)
    matches = mapping.index[mapping.index == asof]
    if len(matches) == 0:
        raise KeyError(f"mapping.pkl に asof_date={asof.date()} の行がありません。")
    if len(matches) > 1:
        raise ValueError(f"mapping.pkl に asof_date={asof.date()} の重複行があります。")

    row = mapping.loc[matches[0]].dropna()
    pairs = pd.DataFrame({"code4": row.index.map(_format_code4), "TSCODE": row.map(_normalise_tscode).to_numpy()})
    pairs = pairs[pairs["TSCODE"] != ""].copy()
    pairs["security_id"] = pairs["code4"] + ".T"

    dup_tscode = pairs.loc[pairs["TSCODE"].duplicated(keep=False), "TSCODE"].unique().tolist()
    if dup_tscode:
        raise ValueError(f"mapping.pkl の当日行で同一TSCODEが複数の4桁コードに対応しています: {dup_tscode[:10]}")
    dup_sid = pairs.loc[pairs["security_id"].duplicated(keep=False), "security_id"].unique().tolist()
    if dup_sid:
        raise ValueError(f"mapping.pkl の当日行でsecurity_idが重複しています: {dup_sid[:10]}")
    return pairs


def load_daily_barra_exposure(asof_date: pd.Timestamp, x_dir: str | Path) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    asof = _to_naive_timestamp(asof_date)
    path = Path(x_dir) / f"X_{asof:%Y%m%d}.pkl"
    if not path.exists():
        raise FileNotFoundError(path)

    payload = pd.read_pickle(path)
    if not isinstance(payload, dict) or "date" not in payload or "X" not in payload:
        raise ValueError("X_YYYYMMDD.pkl must contain {'date': Timestamp, 'X': DataFrame}.")
    payload_date = _to_naive_timestamp(payload["date"])
    if payload_date != asof:
        raise ValueError(f"pkl内date={payload_date.date()} が asof_date={asof.date()} と一致しません。")

    X = payload["X"]
    if not isinstance(X, pd.DataFrame):
        raise TypeError("payload['X'] must be a pandas DataFrame.")
    X = X.copy()
    X.index = X.index.map(_normalise_tscode)
    X.columns = X.columns.map(str)
    X = X.apply(pd.to_numeric, errors="coerce")
    if X.empty:
        raise ValueError(f"{path} のXが空です。")
    meta = {"x_path": path, "pkl_date": payload_date, "raw_tscode_count": X.shape[0], "factor_count": X.shape[1]}
    return X.astype(float), meta


def load_raw_inputs(paths: Mapping[str, str], cfg: ThemeExposureConfig) -> Dict[str, Any]:
    return {
        "stock_returns": _read_table(paths["stock_returns"], wide=True),
        "theme_basket_weights": _read_table(paths["theme_basket_weights"]),
        "barra_estimation_universe": _read_table(paths["barra_estimation_universe"]),
        "mapping": load_mapping(cfg.mapping_path),
    }


def validate_and_normalise_inputs(raw: Mapping[str, Any], cfg: ThemeExposureConfig) -> Dict[str, Any]:
    cfg.validate()

    stock_returns = _normalise_wide(raw["stock_returns"], "stock_returns")

    theme_basket_weights = _normalise_long_dates(raw["theme_basket_weights"], ["effective_date", "available_date"])
    required_bw = {"effective_date", "theme_id", "security_id", "weight"}
    missing = required_bw - set(theme_basket_weights.columns)
    if missing:
        raise ValueError(f"theme_basket_weights is missing columns: {sorted(missing)}")
    if "available_date" not in theme_basket_weights.columns:
        theme_basket_weights["available_date"] = theme_basket_weights["effective_date"]
    else:
        theme_basket_weights["available_date"] = theme_basket_weights["available_date"].fillna(
            theme_basket_weights["effective_date"]
        )
    theme_basket_weights["knowledge_date"] = theme_basket_weights[["effective_date", "available_date"]].max(axis=1)
    theme_basket_weights["theme_id"] = theme_basket_weights["theme_id"].astype(str)
    theme_basket_weights["security_id"] = theme_basket_weights["security_id"].map(_normalise_security_id)
    theme_basket_weights["weight"] = pd.to_numeric(theme_basket_weights["weight"], errors="coerce")
    if theme_basket_weights["weight"].isna().any():
        raise ValueError("theme_basket_weights.weight contains nonnumeric values.")
    if (theme_basket_weights["weight"] < -1e-12).any():
        raise ValueError("Theme basket weights must be nonnegative.")
    theme_basket_weights = theme_basket_weights[theme_basket_weights["weight"] > 0].copy()
    sums = theme_basket_weights.groupby(["knowledge_date", "theme_id"])["weight"].transform("sum")
    if (sums <= 0).any():
        raise ValueError("At least one theme snapshot has nonpositive total weight.")
    theme_basket_weights["weight"] = theme_basket_weights["weight"] / sums

    universe = _normalise_long_dates(raw["barra_estimation_universe"], ["date"])
    required_eu = {"date", "security_id", "in_universe"}
    missing = required_eu - set(universe.columns)
    if missing:
        raise ValueError(f"barra_estimation_universe is missing columns: {sorted(missing)}")
    universe["security_id"] = universe["security_id"].map(_normalise_security_id)
    universe["in_universe"] = _coerce_bool(universe["in_universe"])

    return {
        "stock_returns": stock_returns,
        "theme_basket_weights": theme_basket_weights,
        "barra_estimation_universe": universe,
        "mapping": raw["mapping"],
    }


## Point-in-time スナップショット

`BasketStore` はテーマごとに `knowledge_date <= asof_date` の最新構成だけを使う。Universe は今回の要件に合わせて `asof_date` の行を厳密に使い、直近日へのforward-fillはしない。


In [ ]:
class UniverseStore:
    def __init__(self, universe: pd.DataFrame):
        self.by_date: Dict[pd.Timestamp, pd.Series] = {}
        for d, g in universe.groupby("date", sort=True):
            if g["security_id"].duplicated().any():
                dup = g.loc[g["security_id"].duplicated(keep=False), "security_id"].unique().tolist()
                raise ValueError(f"Universe has duplicated security_id on {pd.Timestamp(d).date()}: {dup[:10]}")
            s = g.set_index("security_id")["in_universe"].astype(bool)
            self.by_date[pd.Timestamp(d)] = s
        self.dates = pd.DatetimeIndex(sorted(self.by_date))
        if len(self.dates) == 0:
            raise ValueError("UniverseStore requires at least one date.")

    def get_exact(self, asof: pd.Timestamp) -> pd.Series:
        asof = _to_naive_timestamp(asof)
        if asof not in self.by_date:
            raise KeyError(f"barra_estimation_universe に asof_date={asof.date()} の行がありません。")
        return self.by_date[asof]


class BasketStore:
    def __init__(self, basket_weights: pd.DataFrame):
        self.versions: Dict[str, list[Tuple[pd.Timestamp, pd.Series]]] = {}
        grouped = basket_weights.groupby(["theme_id", "knowledge_date"], sort=True)
        for (theme, d), g in grouped:
            s = g.groupby("security_id")["weight"].sum()
            s = s[s > 0]
            if s.sum() <= 0:
                continue
            s = s / s.sum()
            self.versions.setdefault(str(theme), []).append((pd.Timestamp(d), s.astype(float)))
        for theme in self.versions:
            self.versions[theme].sort(key=lambda x: x[0])
        self.themes = sorted(self.versions)

    def snapshot_with_dates(self, asof: pd.Timestamp) -> Tuple[pd.DataFrame, pd.Series]:
        asof = _to_naive_timestamp(asof)
        cols = {}
        dates = {}
        for theme, versions in self.versions.items():
            version_dates = pd.DatetimeIndex([v[0] for v in versions])
            pos = version_dates.searchsorted(asof, side="right") - 1
            if pos >= 0:
                cols[theme] = versions[pos][1]
                dates[theme] = versions[pos][0]
        if not cols:
            return pd.DataFrame(dtype=float), pd.Series(dtype="datetime64[ns]", name="basket_knowledge_date")
        a = pd.DataFrame(cols).fillna(0.0)
        sums = a.sum(axis=0)
        a = a.loc[:, sums > 0]
        a = a.div(a.sum(axis=0), axis=1).sort_index()
        date_series = pd.Series(dates, name="basket_knowledge_date").reindex(a.columns)
        return a, pd.to_datetime(date_series)


@dataclass
class Stores:
    estimation_universe: UniverseStore
    baskets: BasketStore


def build_stores(data: Mapping[str, Any]) -> Stores:
    return Stores(
        estimation_universe=UniverseStore(data["barra_estimation_universe"]),
        baskets=BasketStore(data["theme_basket_weights"]),
    )


## テーマ・エクスポージャ作成ロジック

元 notebook の `build_raw_theme_exposure` と同じく、`basket_weight`、`membership`、`rank_weight` を選べる。今回はOLS前提なので、列ごとに単純なユークリッドノルムで単位化する。


In [ ]:
def map_barra_exposure_to_security_id(
    X_tscode: pd.DataFrame,
    mapping: pd.DataFrame,
    asof_date: pd.Timestamp,
) -> Tuple[pd.DataFrame, pd.DataFrame, Dict[str, Any]]:
    pairs = mapping_pairs_for_date(mapping, asof_date)
    tscode_to_security = pairs.set_index("TSCODE")["security_id"]

    tscode_index = pd.Index(X_tscode.index.map(_normalise_tscode), name="TSCODE")
    mapped_security = tscode_to_security.reindex(tscode_index)
    mapped_mask = mapped_security.notna().to_numpy()
    unmapped = pd.Index(tscode_index[~mapped_mask], name="unmapped_TSCODE")

    X_mapped = X_tscode.loc[mapped_mask].copy()
    X_mapped.index = pd.Index(mapped_security[mapped_mask].to_numpy(), name="security_id")
    if X_mapped.index.duplicated().any():
        dup = X_mapped.index[X_mapped.index.duplicated()].unique().tolist()
        raise ValueError(f"mapping後のBarraエクスポージャでsecurity_idが重複しています: {dup[:10]}")

    map_diag = {
        "mapping_date": _to_naive_timestamp(asof_date),
        "mapping_pair_count": len(pairs),
        "x_tscode_count": len(tscode_index),
        "mapping_success_count": int(mapped_mask.sum()),
        "mapping_failure_count": int((~mapped_mask).sum()),
        "mapped_security_id_is_unique": not X_mapped.index.duplicated().any(),
        "unmapped_tscodes": unmapped,
    }
    return X_mapped.sort_index(), pairs, map_diag


def build_raw_theme_exposure(
    basket_snapshot: pd.DataFrame,
    universe: Sequence[str],
    mode: str,
) -> pd.DataFrame:
    a = basket_snapshot.reindex(index=list(universe), fill_value=0.0).astype(float)
    if mode == "membership":
        x = (a > 0).astype(float)
    elif mode == "rank_weight":
        x = a.rank(axis=0, pct=True, method="average").where(a > 0, 0.0)
    elif mode == "basket_weight":
        x = a.copy()
    else:
        raise ValueError(f"Unknown theme_exposure_mode={mode}")

    for col in x.columns:
        v = x[col].to_numpy(dtype=float)
        valid = np.isfinite(v)
        if valid.sum() == 0:
            x[col] = 0.0
            continue
        norm = math.sqrt(max(float(np.sum(v[valid] ** 2)), 0.0))
        if norm > 0:
            x[col] = v / norm
    return x


def ols_column_norms(frame: pd.DataFrame) -> pd.Series:
    out = {}
    for col in frame.columns:
        v = frame[col].to_numpy(dtype=float)
        valid = np.isfinite(v)
        out[col] = math.sqrt(max(float(np.sum(v[valid] ** 2)), 0.0)) if valid.any() else np.nan
    return pd.Series(out, dtype=float, name="ols_norm")


def construct_theme_exposure_snapshot(
    data: Mapping[str, Any],
    stores: Stores,
    cfg: ThemeExposureConfig,
) -> Dict[str, Any]:
    cfg.validate()
    asof = _to_naive_timestamp(cfg.asof_date)

    X_tscode, x_meta = load_daily_barra_exposure(asof, cfg.x_dir)
    X_full, mapping_pairs, mapping_diag = map_barra_exposure_to_security_id(X_tscode, data["mapping"], asof)

    universe = stores.estimation_universe.get_exact(asof)
    in_universe = universe[universe].index
    A_raw, basket_dates = stores.baskets.snapshot_with_dates(asof)
    if A_raw.empty:
        raise ValueError(f"No theme baskets available on or before {asof}.")

    return_universe = pd.Index(data["stock_returns"].columns)
    stock_x_overlap = return_universe.intersection(X_full.index)
    stock_x_universe = stock_x_overlap.intersection(in_universe)

    X_candidate = X_full.reindex(stock_x_universe)
    complete_x = X_candidate.notna().all(axis=1)
    eligible_secs = stock_x_universe[complete_x.to_numpy()]
    X = X_candidate.loc[eligible_secs]

    original_sum = A_raw.sum(axis=0).replace(0.0, np.nan)
    A_before_filter = A_raw.reindex(index=eligible_secs, fill_value=0.0)
    coverage_all = (A_before_filter.sum(axis=0) / original_sum).replace([np.inf, -np.inf], np.nan)

    keep_themes = coverage_all[coverage_all >= cfg.min_basket_coverage].index
    A_t = A_before_filter.loc[:, keep_themes]
    A_t = A_t.loc[:, A_t.sum(axis=0) > 0]
    if A_t.empty:
        raise ValueError("No theme passes min_basket_coverage after universe/exposure filters.")
    A_t = A_t.div(A_t.sum(axis=0), axis=1)
    coverage = coverage_all.reindex(A_t.columns)

    X_M_t = build_raw_theme_exposure(A_t, eligible_secs, cfg.theme_exposure_mode)
    regression_weights = pd.Series(1.0, index=eligible_secs, name="regression_weight")

    filter_counts = pd.Series(
        {
            "x_raw_tscode_rows": x_meta["raw_tscode_count"],
            "x_mapped_security_rows": len(X_full),
            "mapping_success_count": mapping_diag["mapping_success_count"],
            "mapping_failure_count": mapping_diag["mapping_failure_count"],
            "stock_return_columns": len(return_universe),
            "in_universe_true": len(in_universe),
            "stock_and_x_overlap": len(stock_x_overlap),
            "stock_x_universe_overlap": len(stock_x_universe),
            "dropped_for_missing_barra_exposure": int((~complete_x).sum()),
            "complete_barra_exposure": len(eligible_secs),
            "raw_basket_security_count": A_raw.shape[0],
            "eligible_basket_security_count": int((A_raw.index.isin(eligible_secs)).sum()),
        },
        name="n_securities",
    )

    theme_filter = pd.DataFrame(
        {
            "raw_weight_sum_before_filter": original_sum,
            "coverage_after_security_filters": coverage_all,
            "kept": pd.Series(A_raw.columns.isin(A_t.columns), index=A_raw.columns),
            "basket_knowledge_date": basket_dates.reindex(A_raw.columns),
        }
    )
    theme_filter.index.name = "theme_id"

    date_diagnostics = pd.Series(
        {
            "asof_date": asof,
            "x_path": str(x_meta["x_path"]),
            "pkl_date": x_meta["pkl_date"],
            "mapping_path": str(Path(cfg.mapping_path)),
            "mapping_date": mapping_diag["mapping_date"],
            "latest_basket_knowledge_date_used": basket_dates.reindex(A_t.columns).max(),
            "raw_theme_count": A_raw.shape[1],
            "kept_theme_count": A_t.shape[1],
            "factor_count": x_meta["factor_count"],
        },
        name="value",
    )

    diagnostics = {
        "date": date_diagnostics,
        "security_filter_counts": filter_counts,
        "theme_filter": theme_filter,
        "A_column_sum": A_t.sum(axis=0).rename("A_column_sum"),
        "X_M_ols_norm": ols_column_norms(X_M_t),
        "mapping_pairs": mapping_pairs,
        "unmapped_tscodes": mapping_diag["unmapped_tscodes"],
        "barra_factor_missing_by_security": X_candidate.isna().sum(axis=1).rename("n_missing_factors"),
    }

    return {
        "asof_date": asof,
        "X": X,
        "A_raw": A_raw,
        "A_t": A_t,
        "coverage": coverage.rename("coverage"),
        "X_M_t": X_M_t,
        "regression_weights": regression_weights,
        "eligible_securities": pd.Index(eligible_secs),
        "diagnostics": diagnostics,
    }


## 実行セル

`cfg.asof_date`、`cfg.x_dir`、`cfg.mapping_path`、`paths` を設定してから実行する。


In [ ]:
raw_inputs = load_raw_inputs(paths, cfg)
data = validate_and_normalise_inputs(raw_inputs, cfg)
stores = build_stores(data)

snapshot = construct_theme_exposure_snapshot(data, stores, cfg)

A_t = snapshot["A_t"]
coverage = snapshot["coverage"]
X_M_t = snapshot["X_M_t"]
X = snapshot["X"]
regression_weights = snapshot["regression_weights"]
diagnostics = snapshot["diagnostics"]

print(f"asof_date: {snapshot['asof_date'].date()}")
print(f"A_t shape: {A_t.shape}")
print(f"X_M_t shape: {X_M_t.shape}")
print(f"X shape: {X.shape}")


## 出力確認

`A_t` は実際に売買対象となるテーマバスケット構成行列、`X_M_t` は項目1で構築するテーマ・エクスポージャ行列である。ここでは両者は同じ列集合を持つ。


In [ ]:
display(diagnostics["date"].to_frame())
display(diagnostics["security_filter_counts"].to_frame())

if len(diagnostics["unmapped_tscodes"]):
    display(pd.Series(diagnostics["unmapped_tscodes"], name="unmapped_TSCODE").head(30).to_frame())

display(diagnostics["theme_filter"].sort_values(["kept", "coverage_after_security_filters"], ascending=[False, False]))
display(pd.concat([diagnostics["A_column_sum"], coverage, diagnostics["X_M_ols_norm"]], axis=1))

display(A_t.head())
display(X_M_t.head())


## 受入チェック

次のチェックが通れば、項目1としては期待どおりである。

- `X_YYYYMMDD.pkl` の `date` が `asof_date` と一致
- `mapping.pkl` の当日行を使用
- すべての `X.index=TSCODE` が `security_id=4桁.T` へ変換可能
- `in_universe=False` の銘柄は除外
- `A_t` の各テーマ列合計が1
- 残ったテーマは `coverage >= min_basket_coverage`
- `X_M_t` の各テーマ列がOLSノルムで1


In [ ]:
tol = 1e-8
asof = snapshot["asof_date"]

assert not A_t.empty, "A_t is empty."
assert not X_M_t.empty, "X_M_t is empty."
assert A_t.index.equals(X_M_t.index), "A_t and X_M_t must have the same security index."
assert A_t.columns.equals(X_M_t.columns), "A_t and X_M_t must have the same theme columns."
assert diagnostics["date"].loc["pkl_date"] == asof, "pkl内date must equal asof_date."
assert diagnostics["date"].loc["mapping_date"] == asof, "mapping date must equal asof_date."
assert diagnostics["security_filter_counts"].loc["mapping_failure_count"] == 0, "All X.index TSCODE values must map to security_id."
assert np.allclose(A_t.sum(axis=0).to_numpy(dtype=float), 1.0, atol=tol), "A_t columns must sum to 1."
assert (coverage >= cfg.min_basket_coverage - tol).all(), "All kept themes must pass min_basket_coverage."
assert np.allclose(
    diagnostics["X_M_ols_norm"].dropna().to_numpy(dtype=float),
    1.0,
    atol=1e-8,
), "X_M_t columns must have OLS unit norm."

universe_today = stores.estimation_universe.get_exact(asof)
assert universe_today.reindex(A_t.index).fillna(False).all(), "A_t contains securities outside in_universe=True."
kept_theme_dates = diagnostics["theme_filter"].loc[A_t.columns, "basket_knowledge_date"]
assert (kept_theme_dates <= asof).all(), "Future basket information was used."

print("Item 1 acceptance checks passed.")


## 次の項目へ進む前の確認メモ

項目2では、この notebook で作った `X_M_t` を当日Barraエクスポージャ `X` へ回帰し、純化テーマ・エクスポージャ `Z_t = X_{M\perp V,t}` を作る。今回の前提ではOLSなので、その回帰の重みは `regression_weights=1.0` を使う。

この notebook では項目1に限定するため、Barra純化、OLS係数、OLS残差推定は実行しない。
